# nb_03b — Gold: `fact_workforce_event` with as-of key resolution

**Module 3 (fact).** The payoff of SCD2: we attribute each pay-setting event to
the **pay grid in force on the event date**, then compute a **compa-ratio** we can
compare against *today's* re-benchmarked grid.

Resolved per event:
- `date_key` → `dim_date`
- `cost_center_key` → `dim_cost_center` (SCD1, direct)
- `worker_key` → `dim_worker` **as-of** `event_date` (SCD2 range join)
- `pay_band_key` → `dim_pay_band` **as-of** on (group, level, event_date) (SCD2 range join)

We precompute band bounds and compa-ratio at event so the model's DAX stays simple.

In [ ]:
from pyspark.sql import functions as F
ev  = spark.table("silver.workforce_event")
dcc = spark.table("gold.dim_cost_center").select("cost_center_key","cost_center_id")
dw  = spark.table("gold.dim_worker").select("worker_key","employee_id",
        F.col("effective_from").alias("w_from"), F.col("effective_to").alias("w_to"))
dpb = spark.table("gold.dim_pay_band").select("pay_band_key",
        F.col("classification_group").alias("pb_group"),
        F.col("classification_level").alias("pb_level"),
        "band_min","band_mid","band_max","effective_from","effective_to")
print(f"silver events to load: {ev.count():,}")

In [ ]:
COMP = ["Hire","Promotion","Step Increment"]
fact = (ev
    .withColumn("date_key", F.date_format("event_date","yyyyMMdd").cast("int"))
    .join(dcc, "cost_center_id", "left")
    # as-of worker (SCD2 range join)
    .join(dw, (ev.employee_id==dw.employee_id) &
              (ev.event_date>=dw.w_from) & (ev.event_date<=dw.w_to), "left")
    # as-of pay band (SCD2 range join on group+level+date)
    .join(dpb, (ev.classification_group==dpb.pb_group) &
               (ev.classification_level==dpb.pb_level) &
               (ev.event_date>=dpb.effective_from) &
               (ev.event_date<=dpb.effective_to), "left")
    # measures
    .withColumn("base_salary_cad",
        F.when(F.col("event_type").isin(COMP), F.col("amount_cad")))
    .withColumn("bonus_cad",
        F.when(F.col("event_type")=="Performance Pay", F.col("amount_cad")))
    .withColumn("compa_ratio_at_event",
        F.when(F.col("base_salary_cad").isNotNull() & (F.col("band_mid")>0),
               F.round(F.col("base_salary_cad")/F.col("band_mid"),4)))
    .withColumn("below_band_at_event",
        F.when(F.col("base_salary_cad").isNotNull(),
               F.col("base_salary_cad") < F.col("band_min")))
    .select("event_id","date_key","cost_center_key","worker_key","pay_band_key",
            "classification_group","classification_level","event_type",
            "amount_cad","base_salary_cad","bonus_cad",
            F.col("band_min").alias("band_min_at_event"),
            F.col("band_mid").alias("band_mid_at_event"),
            F.col("band_max").alias("band_max_at_event"),
            "compa_ratio_at_event","below_band_at_event",
            "local_currency","source_system","ingest_ts"))
(fact.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.fact_workforce_event"))
print(f"fact rows: {fact.count():,}")

## Data-quality gate — no unresolved keys should remain

In [ ]:
spark.sql("""
  SELECT sum(case when worker_key    is null then 1 else 0 end) AS null_worker,
         sum(case when pay_band_key  is null then 1 else 0 end) AS null_pay_band,
         sum(case when cost_center_key is null then 1 else 0 end) AS null_cost_center,
         count(*) AS total
  FROM gold.fact_workforce_event""").show()

## The SCD2 payoff — compa-ratio as-was vs as-is

Average compa-ratio of pay set in 2021, measured against the grid **in force then**
(as-was) — every such pay looks healthy near 1.0. But the grid has since been
re-benchmarked upward, so the same salaries sit lower against **today's** grid.
The as-is view is a model measure (Module 4); here's the as-was baseline that only
the versioned `pay_band_key` makes possible.

In [ ]:
spark.sql("""
  SELECT d.year AS pay_set_year,
         round(avg(f.compa_ratio_at_event),3) AS avg_compa_ratio_as_was,
         sum(case when f.below_band_at_event then 1 else 0 end) AS count_below_band_as_was
  FROM gold.fact_workforce_event f
  JOIN gold.dim_date d ON f.date_key = d.date_key
  WHERE f.base_salary_cad IS NOT NULL
  GROUP BY d.year ORDER BY d.year""").show()